In [ ]:
# Installation automatique des dépendances requises dans le noyau Jupyter actuel
%pip install -r ../requirements.txt

# 📊 Jalon 1 : Analyse Exploratoire des Données (EDA) & Visualisation (Squelette Étudiant)

Ce notebook est dédié à la découverte de relations clés et à l'analyse visuelle de nos données. À partir du jeu de données propre généré précédemment, nous allons enrichir nos variables explicatives et appeler les fonctions de notre module de visualisation `src.utils_viz` pour générer des graphiques professionnels.

### 1. Importation des packages et configuration du style

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath('..'))

%matplotlib inline

In [2]:
# Chargement du dataset propre
df = pd.read_csv('../data/processed/owid-monkeypox-data_clean.csv')
df['date'] = pd.to_datetime(df['date'])
df.head()

,country,date,total_cases,total_deaths,daily_new_cases,daily_new_deaths
0,Andorra,2022-07-25,2.0,0.0,2.0,0.0
1,Andorra,2022-07-26,3.0,0.0,1.0,0.0
2,Andorra,2022-07-27,3.0,0.0,0.0,0.0
3,Andorra,2022-07-28,3.0,0.0,0.0,0.0
4,Andorra,2022-07-29,3.0,0.0,0.0,0.0


### 2. Ingénierie de variables temporelles

**À faire par l'étudiant :**
Appliquez la fonction `feature_engineering` de `src.data_clean` pour enrichir votre DataFrame en caractéristiques de temps classiques .

In [3]:
df_feat = df.copy()

df_feat['year'] = df_feat['date'].dt.year
df_feat['month'] = df_feat['date'].dt.month
df_feat['year_month'] = df_feat['date'].dt.to_period('M').astype(str)
df_feat['dayofweek'] = df_feat['date'].dt.dayofweek

df_feat.head()

,country,date,total_cases,total_deaths,daily_new_cases,daily_new_deaths,year,month,year_month,dayofweek
0,Andorra,2022-07-25,2.0,0.0,2.0,0.0,2022,7,2022-07,0
1,Andorra,2022-07-26,3.0,0.0,1.0,0.0,2022,7,2022-07,1
2,Andorra,2022-07-27,3.0,0.0,0.0,0.0,2022,7,2022-07,2
3,Andorra,2022-07-28,3.0,0.0,0.0,0.0,2022,7,2022-07,3
4,Andorra,2022-07-29,3.0,0.0,0.0,0.0,2022,7,2022-07,4


### 3. Visualisations Professionnelles

#### A. Profils d'évolution et tendances


Cette figure montre l’évolution du nombre total de cas dans cinq pays au cours du temps. On observe d’abord une forte montée des cas à partir de l’été 2022, avec des rythmes différents selon les pays, puis une stabilisation progressive vers le début de l’année 2023.

Un premier constat est que les États-Unis présentent le niveau de cas le plus élevé sur toute la période, avec une croissance rapide puis un plateau autour de 30 000 cas. Le Brésil arrive ensuite avec une progression plus lente mais continue, tandis que l’Espagne atteint également un niveau important avant de se stabiliser. La France et la Colombie affichent des volumes plus faibles, avec une montée plus tardive et une stabilisation autour de 4 000 cas.

Cette figure permet donc de voir que la propagation du Mpox n’a pas été uniforme selon les pays. Elle met en évidence des dynamiques très différentes, ce qui justifie une analyse par pays plutôt qu’une lecture globale unique.


In [ ]:
top_5_countries = (
    df_feat.groupby('country')['total_cases']
    .max()
    .sort_values(ascending=False)
    .head(5)
    .index
)

df_top5 = df_feat[df_feat['country'].isin(top_5_countries)]

fig1 = uv.plot_generic_trends(
    df_top5,
    x_col='date',
    y_col='total_cases',
    group_col='country'
)

plt.show()

NameError: name 'plot_generic_trends' is not defined

#### B. Répartition mondiale des cas cumulés de mpox par pays

Cette carte permet de visualiser la répartition géographique des cas cumulés de mpox.  
Elle est pertinente car le dataset contient une dimension spatiale avec une variable pays.  
Contrairement à un simple tableau, la carte facilite l’identification rapide des zones les plus touchées et permet de comparer visuellement l’intensité de l’épidémie entre les pays.

In [18]:
import plotly.express as px

# On récupère le nombre maximum de cas cumulés par pays
map_data = (
    df_feat.groupby("country", as_index=False)["total_cases"]
    .max()
    .sort_values(by="total_cases", ascending=False)
)

fig = px.choropleth(
    map_data,
    locations="country",
    locationmode="country names",
    color="total_cases",
    hover_name="country",
    color_continuous_scale="Reds",
    title="Répartition mondiale des cas cumulés de mpox par pays"
)

fig.update_layout(
    title_x=0.5,
    geo=dict(showframe=False, showcoastlines=True)
)

fig.show()

#### C. Carte animée de la propagation du mpox

Ce graphique montre la propagation du Mpox dans le monde au fil des mois, en colorant chaque pays selon le nombre de cas cumulés. L’animation temporelle permet de voir l’évolution pays par pays, tandis que les infos au survol affichent les cas cumulés, les nouveaux cas mensuels et la période observée.

Comme le dataset contient une dimension temporelle (`year_month`) et une dimension géographique (`country`), une carte animée est pertinente pour analyser la propagation de l’épidémie entre les pays.

In [24]:
import plotly.express as px

# Préparation des données mensuelles par pays
map_time_data = (
    df_feat.groupby(["country", "year_month"], as_index=False)
    .agg({
        "total_cases": "max",
        "daily_new_cases": "sum"
    })
)

# Correction des noms de pays pour Plotly
map_time_data["country_map"] = map_time_data["country"].str.replace("_", " ")

# Conversion du mois en date pour garantir l'ordre chronologique
map_time_data["year_month_date"] = pd.to_datetime(map_time_data["year_month"])

# Tri chronologique
map_time_data = map_time_data.sort_values("year_month_date")

fig = px.choropleth(
    map_time_data,
    locations="country_map",
    locationmode="country names",
    color="total_cases",
    hover_name="country",
    hover_data={
        "total_cases": True,
        "daily_new_cases": True,
        "year_month": True,
        "country_map": False
    },
    animation_frame="year_month",
    color_continuous_scale="Reds",
    title="Propagation du mpox dans le monde au fil du temps"
)

fig.update_layout(
    title_x=0.5,
    geo=dict(
        showframe=False,
        showcoastlines=True
    )
)

fig.show()

### 4. Synthèse des observations clés

L’analyse exploratoire du dataset mpox met en évidence une forte dimension à la fois temporelle et géographique.

Le premier graphique, qui présente l’évolution des cas cumulés pour les cinq pays les plus touchés, montre que la progression de l’épidémie n’est pas identique selon les pays. Certains pays connaissent une croissance rapide du nombre de cas, tandis que d’autres présentent une évolution plus progressive. Cela confirme l’intérêt d’une analyse par pays plutôt qu’une analyse uniquement globale.


La carte animée de propagation apporte une lecture complémentaire en combinant la dimension spatiale et temporelle. Elle permet de visualiser l’évolution de l’épidémie dans le monde au fil des mois et d’identifier les zones où les cas apparaissent ou augmentent progressivement. L’agrégation mensuelle rend cette visualisation plus lisible que les données quotidiennes, souvent irrégulières.

Dans l’ensemble, les visualisations montrent que les variables les plus pertinentes pour analyser ce dataset sont `date`, `country`, `total_cases`, `daily_new_cases`, `total_deaths` et `daily_new_deaths`. Elles permettent de comprendre l’évolution de l’épidémie, d’identifier les pays les plus touchés et d’observer la diffusion géographique du mpox.

Cette première EDA confirme donc que le dataset est adapté à une analyse temporelle et géographique de l’épidémie. Elle fournit une base solide pour produire des indicateurs de suivi et approfondir l’analyse dans les étapes suivantes du projet.